<a href="https://colab.research.google.com/github/ravikeerthi7606/HousePricePrediction/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q mlflow joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
#mlflow directory
import os
from pathlib import Path

# Opt-in to filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow_dir = Path(
    "/content/drive/MyDrive/ML project/HousePricePrediction/mlruns"
)

mlflow_dir.mkdir(parents=True, exist_ok=True)

#Create an experiment
import mlflow

mlflow.set_tracking_uri(mlflow_dir.as_uri())

mlflow.set_experiment("House Price Prediction")

2026/10/05 08:49:16 INFO mlflow.tracking.fluent: Experiment with name 'House Price Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///content/drive/MyDrive/ML%20project/HousePricePrediction/mlruns/769246350787289816', creation_time=1791190156868, effective_trace_archival_retention=None, experiment_id='769246350787289816', last_update_time=1791190156868, lifecycle_stage='active', name='House Price Prediction', tags={}, trace_location=None, workspace='default'>

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ames-housing-dataset' dataset.
Path to dataset files: /kaggle/input/ames-housing-dataset


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#DON'T FORGET TO CHANGE THE PATH
df = pd.read_csv('/kaggle/input/ames-housing-dataset/AmesHousing.csv')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

In [35]:
X = df.drop(columns='SalePrice')
y = df['SalePrice']

numeric_columns = X.select_dtypes(include=np.number).columns
categorical_columns = X.select_dtypes(exclude=np.number).columns


print(f'numerical columns: {len(numeric_columns)}')
print(f'categorical columns: {len(categorical_columns)}')

numerical columns: 37
categorical columns: 43


In [36]:
#imputation
numeric_pipeline = Pipeline(
    steps=[('impute', SimpleImputer(strategy='median')),
           ('scale', StandardScaler())]
)
categorical_pipeline = Pipeline(
    steps=[('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))]
  )

#column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns)
    ])

In [37]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

In [46]:
#Linear regression model
linear_model = Ridge()

#Pipeline
linear_pipeline = Pipeline(
    steps=[('preprocessor', preprocessor),
    ('linear_model',linear_model)]
)

In [47]:
#mlflow
from pathlib import Path

with mlflow.start_run(run_name='Ridge'):
  linear_pipeline.fit(X_train, y_train)

  rmse = np.sqrt(mean_squared_error(y_test, linear_pipeline.predict(X_test)))
  mae = mean_absolute_error(y_test, linear_pipeline.predict(X_test))
  r2 = r2_score(y_test, linear_pipeline.predict(X_test))

  # Ensure target directory exists
  model_path = Path("/content/drive/MyDrive/ML project/HousePricePrediction/models/Ridge.joblib")
  model_path.parent.mkdir(parents=True, exist_ok=True)

  joblib.dump(linear_pipeline, model_path)

  mlflow.sklearn.log_model(
    linear_pipeline,
    "Ridge",
    serialization_format='cloudpickle'
  )

  mlflow.log_metric('mse', rmse)
  mlflow.log_metric('mae', mae)
  mlflow.log_metric('r2', r2)

2026/10/05 09:06:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/05 09:06:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [48]:
#MAE
print(mean_absolute_error(y_test, linear_pipeline.predict(X_test)))
#mse
print(np.sqrt(mean_squared_error(y_test, linear_pipeline.predict(X_test))))
#r2
print(r2_score(y_test, linear_pipeline.predict(X_test)))

16971.016574274097
31238.351452024956
0.8646223762906277


In [49]:
experiment = mlflow.get_experiment_by_name(
    "House Price Prediction"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.r2,metrics.mse,metrics.mae,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.type
0,40e9a620caab43cba581756300c29f2c,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 09:06:56.801000+00:00,2026-10-05 09:07:02.833000+00:00,0.864622,31238.351452,16971.016574,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,root,Ridge,NOTEBOOK
1,f1b38b33b73048108945549c23cadc4d,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:59:35.693000+00:00,2026-10-05 08:59:43.164000+00:00,0.853538,32492.015632,16891.331812,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,root,Linear Regression,NOTEBOOK
2,34960b86d2524cb9b578f539b26012cf,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:59:18.545000+00:00,2026-10-05 08:59:19.701000+00:00,NaN,NaN,NaN,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,root,Linear Regression,NOTEBOOK
3,92bd3842931b45a9a95fc2768580c3a3,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:58:30.262000+00:00,2026-10-05 08:58:30.933000+00:00,NaN,NaN,NaN,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,root,Linear Regression,NOTEBOOK
